# Create ResBaz Eventbrite events
This script connects to `main-sheet` to create Eventbrite events based on data within the sessions sheet.

## 0 - Import libraries and authenticate google account

In [ ]:
# import/install packages
!pip install ruamel.yaml
import pandas as pd
# import yaml
from ruamel.yaml import YAML
yaml = YAML()
import json
import requests
from pprint import pprint
from google.colab import auth
import gspread
from google.auth import default
from pprint import pprint
from tqdm.auto import tqdm
from tqdm.contrib.concurrent import thread_map
import sys
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
from google.colab import userdata

In [ ]:
# auth google
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

## 1 - Access `main_sheet`

In [ ]:
# access sessions and schedule sheets and return as dataframe
main_sheet = gc.open_by_key(userdata.get('main_sheet_key'))
sessions_sheet = main_sheet.worksheet('sessions')
schedule_sheet = main_sheet.worksheet('schedule')

rows = sessions_sheet.get_all_values() # provides list of rows
sessions_df = pd.DataFrame(rows[1:], columns=rows[0])
orig_sessions_df = sessions_df.copy() # retain original copy of sessions

rows = schedule_sheet.get_all_values()
schedule_df = pd.DataFrame(rows[1:], columns=rows[0])
orig_schedule = schedule_df.copy()

schedule_df

## 2 - Define schedule & update sessions
For an Eventbrite event to be created:
- The event id must be added to the 'schedule' sheet AND the 'status' column must contain 'confirmed'.
- Externally hosted events should have 'external' in the status column, but still be added to the schedule sheet, in order for the schedule.yaml to be created later.

This step defines NZ and UTC timing based on the schedule sheet.  

In [ ]:
# define lookup table containing startTime & endTime kiwi time
def define_session_times(id_col): # return session id and associated times
  for id in id_col:
    if id != '':
      startTime = pd.Timestamp(row.date + "T" + row.startTime, tz="Pacific/Auckland")
      endTime = pd.Timestamp(row.date + "T" + row.endTime, tz="Pacific/Auckland")

      if (int(id_col) in session_lookup) and (session_lookup[int(id_col)]["startTime"]):
        session_lookup[int(id_col)]["endTime"] = endTime
      else:
        session_lookup[int(id_col)] = {
        "startTime" : startTime,
        "endTime" : endTime
        }

session_lookup = {}
for row in schedule_df.itertuples():
  define_session_times(row.track1)
  define_session_times(row.track2)
  define_session_times(row.track3)

# add times to sessions dataframe
def get_session(id, key):
  if id != '':
    id = int(id)
    if id in session_lookup:
      return session_lookup[id].get(key, pd.NA)
  else:
    return pd.NA

sessions_df["start_time_Auckland"] = pd.to_datetime(sessions_df.id.apply(get_session, key="startTime"))
sessions_df["end_time_Auckland"] = pd.to_datetime(sessions_df.id.apply(get_session, key="endTime"))
sessions_df["start_time_UTC"] = sessions_df.start_time_Auckland.dt.tz_convert("UTC")
sessions_df["end_time_UTC"] = sessions_df.end_time_Auckland.dt.tz_convert("UTC")

sessions_df

## 3 - Update sessions sheet
- Update time fields in `sessions` sheet.
- Define dataframe of sessions that are running.

In [ ]:
updates = sessions_df[~sessions_df.start_time_UTC.isna() & (sessions_df.start_time_UTC != orig_sessions_df.start_time_UTC)]

In [ ]:
for i, row in tqdm(updates.iterrows(), total=len(updates)):
  sessions_sheet.update([[row.start_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ')]], f"P{i+2}")
  sessions_sheet.update([[row.end_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ')]], f"Q{i+2}")
  sessions_sheet.update([[row.start_time_Auckland.strftime('%Y-%m-%dT%H:%M:%SZ')]], f"R{i+2}")
  sessions_sheet.update([[row.end_time_Auckland.strftime('%Y-%m-%dT%H:%M:%SZ')]], f"S{i+2}")

In [ ]:
df_sessions_running = sessions_df[~sessions_df.start_time_UTC.isna()]
df_sessions_confirmed = df_sessions_running[df_sessions_running["status"] == "confirmed"]
display(df_sessions_confirmed)

In [ ]:
#test for 1 event
df_sessions_confirmed = df_sessions_confirmed.iloc[0]
df_sessions_confirmed

## 4 - Generate eventbrite sessions
- Eventbrite access requires api key stored as secret.

In [ ]:
# define api url and headers
URL = "https://www.eventbriteapi.com/v3/events/"
headers = {
    "Authorization": f"Bearer {userdata.get('eb_api_key')}",
    "Content-Type": "application/json"
}


In [ ]:
pprint(requests.get(f"{URL}/634393647477/", headers=headers).json()) # test for one session

### Generate eventbrite sessions
- Parallelised for speed
- `200` response is success, `400` is an error

In [ ]:
#define the event that will be copied from, currently = a ResBaz template draft event: 1341387318579
template_event_id = "1341387318579"

In [ ]:
def gen_event(row): # function to create event
 response = requests.post(f"{URL}/{template_event_id}/copy/", headers=headers) # use copy of this event as template
 new_id = response.json()["id"]
 response = requests.post(f"{URL}/{new_id}/", headers=headers, json={
   'event.name.html': row.title + " [ResBaz]",
   'event.description.html': row.description,
   'event.start.utc': row.start_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
   'event.start.timezone': 'Pacific/Auckland',
   'event.end.utc': row.end_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
   'event.end.timezone': 'Pacific/Auckland',
   'event.capacity': row.capacity,
 })
 print(response)
 return response


responses = thread_map(gen_event, df_sessions_confirmed.itertuples(), total=len(df_sessions_confirmed))
responses

In [ ]:
#capture only sucesses (if a second attempt is needed)
failed_responses = [response for response in responses if response.status_code != 200]
assert len(failed_responses) == 0

In [ ]:
event_ids = [r.json()["id"] for r in failed_responses]
event_ids

In [ ]:
# add eventbrite sessions IDs/URLs to sessions df
df_sessions_confirmed["eventbrite_id"] = [r.json()["id"] for r in responses]
df_sessions_confirmed["eventbrite_url"] = [r.json()["url"] for r in responses]

# update google sheet
for i, row in tqdm(df_sessions_confirmed.iterrows(), total=len(df_sessions_confirmed)):
  sessions_sheet.update([[row.eventbrite_url]], f"M{i+2}")

### Publish eventbrite sessions


In [ ]:
def publish_event(event_id):
  response = requests.post(f"{URL}{event_id}/publish/", headers=headers)
  return response

publish_responses = thread_map(publish_event, df_sessions_confirmed.eventbrite_id, total=len(df_sessions_confirmed))
publish_responses

In [ ]:
# Check if an event has been added to sheet after publishing initial wave of events
unpublished_events = []
for event_id in event_ids:
  response = requests.get(f"{URL}/{event_id}/", headers=headers).json()
  if response["status"] != 'live':
    unpublished_events.append(event_id)
unpublished_events

In [ ]:
# publish unpublished events

publish_responses = thread_map(publish_event, unpublished_events, total=len(unpublished_events))

#### Workflow for deleting draft events (Use with CAUTION)
- There are 4 statuses:
  - draft
  - scheduled
  - live
  - deleted


In [ ]:
#filter variables to caputre only our events
cer_organizer_id = "75893560993"
resbaz_logo_id = "1006482323"
template_id = "1341387318579"

In [ ]:
dry_run = True

# delete draft events -
# ---THESE ARE NO LONGER CREATED AS PART OF OUR OWN EVENTBRITE ACCOUNT - WE USE A FACULTY SUBACCOUNT - BE CAREFUL!
# ---CONFIRM THAT ONLY RESBAZ EVENTS ARE CAUGHT BY THE FILTER BEFORE RUNNING WITH DRY RUN OFF--
response = requests.get(f"https://www.eventbriteapi.com/v3/organizations/30350306317/events?status=draft&page_size=200", headers=headers)
events = pd.json_normalize(response.json()["events"])
events_filtered = events[(events['organizer_id'] == cer_organizer_id) & (events["logo_id"] == resbaz_logo_id) & (events["id"] != template_id)]
display(events_filtered)

if not dry_run and len(events_filtered) > 0:#delete drafts (e.g. test)
  def delete_event(id):
    response = requests.delete(f"{URL}{id}", headers=headers)
    return response
  thread_map(delete_event, events_filtered.id)

## 5 - Update session descriptions via 'structured content'.

- EB uses 'structured content' for creating rich event descriptions.  
- structured content is versioned, see instructions in code cells.
-

In [ ]:
# check what structured content is returned
response = requests.get(f"{URL}{template_event_id}/structured_content/edit/", headers=headers)
pprint(response.json())

In [ ]:
#get Id's from sheet (if they are not available from responses)
df_sessions_confirmed["eventbrite_id"] = df_sessions_confirmed.registration_link.str.split("-").str[-1]

In [ ]:
def set_structured_content(row): # if encountering issues, change version of structured content e.g. `.../structured_content/2/` or `.../structured_content/3/`
  get_response = requests.get(f"{URL}{row.eventbrite_id}/structured_content/edit", headers=headers).json()
  #display(get_response)
  version_number = int(get_response["page_version_number"]) +1
  response = requests.post(f"{URL}{row.eventbrite_id}/structured_content/{version_number}/", headers=headers, json={
      "modules": [{
        "data": {
            "body": {
                "alignment": "left",
                "text": row.description
            },
            "type": "text",
        },
        "type": "text"
      }],
      "publish": True,
      'purpose': 'listing',
  })
  return response


#response = set_structured_content(df_sessions_confirmed.iloc[3]) # test on one row
responses = thread_map(set_structured_content, df_sessions_confirmed.itertuples(), total=len(df_sessions_confirmed))
# responses


In [ ]:
def remove_unstructured_description(row):
   response = requests.post(f"{URL}{row.eventbrite_id}/", headers=headers, json={
    'event.description.html': "",
    })
   return response

# response = remove_unstructured_description(df_sessions_confirmed.iloc[1])
# response.content
responses = thread_map(remove_unstructured_description, df_sessions_confirmed.itertuples(), total=len(df_sessions_confirmed))

In [ ]:
# Update EB times - only if times have been changed in the sheet
def update_times(row):
   response = requests.post(f"{URL}{row.eventbrite_id}/", headers=headers, json={
    'event.start.utc': row.start_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'event.start.timezone': 'Pacific/Auckland',
    'event.end.utc': row.end_time_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'event.end.timezone': 'Pacific/Auckland',
    })
   return response

#response = update_times(df_sessions_confirmed.iloc[0])
#response.content
responses = thread_map(update_times, df_sessions_confirmed.itertuples(), total=len(df_sessions_confirmed))

In [ ]:
requests.get(f"{URL}{df_sessions_confirmed.iloc[0].eventbrite_id}", headers=headers).json()

In [ ]:
response

In [ ]:
display(df_sessions_confirmed)

In [ ]:
# check events with errors
status_codes = [r.status_code for r in responses]

error_df = df_sessions_confirmed.loc[df_sessions_confirmed.index[pd.Series(status_codes) != 200]]
error_df

## 6 - Add Zoom links to eventbrite
- Zoom links MUST be manually added to sessions sheet first
- Zoom links are added to EB events via structured content - https://www.eventbrite.com/platform/docs/online-event-page


In [ ]:
def set_zoom_link(row):
  get_response = requests.get(f"{URL}{row.eventbrite_id}/structured_content/edit", headers=headers).json()
  version_number = int(get_response["page_version_number"]) +1
  response = requests.post(f"{URL}{row.eventbrite_id}/structured_content/{version_number}/?purpose=digital_content", headers=headers, json={
      "modules": [{
        "data":{
            "webinar_url":{
              "text":"Zoom link",
              "url": row.zoom_link
            }
        },
        "type": "webinar"
    }],
    "publish": True,
    'purpose': 'digital_content',
  })
  if response.status_code != 200:
     print(f"Hey we got an error with event {row.eventbrite_id} - response code: {response.status_code} {response.content} - do you need to increment the version number?")
  return response

#response = set_zoom_link(df_sessions_confirmed.iloc[3]) # test on one row
repsonses = thread_map(set_zoom_link, df_sessions_confirmed.itertuples(), total=len(df_sessions_confirmed))
# repsonses

In [ ]:
requests.get(f"{URL}{df_sessions_confirmed.iloc[1].eventbrite_id}/structured_content/", headers=headers).json()

In [ ]:
requests.get(f"{URL}{df_sessions_confirmed.iloc[1].eventbrite_id}", headers=headers).json()

In [ ]:
print(response)

## Update ticket end times

In [ ]:
#get ticket classess (currently just from template event)
eventbrite_id = template_event_id
get_response = requests.get(f"{URL}{eventbrite_id}/ticket_classes/", headers=headers).json()
ticket_class_id = get_response['ticket_classes'][0]['id']

json_body = {
  "ticket_class": {
  "name": "General Admit",
  "sales_end": "2025-06-30T23:59:00Z",
  "capacity": 500,
  "free": True,
  "ticket_classes":[{"id":ticket_class_id}]
  }
}

response = requests.post(f"{URL}{eventbrite_id}/ticket_classes/{ticket_class_id}", headers=headers, data=json.dumps(json_body)
)
display(response.status_code)
display(response.text)
display(response.json())